# E4 — Chronos Zero-Shot Baseline for RUL Prediction on C-MAPSS FD001

**Study:** Agentic Multi-Machine Predictive Maintenance (AMMPM) using Time-Series Foundation Models for Explainable and Trustworthy RUL Prediction.

**Purpose:** This is the experiment the whole TSFM half of the study hinges on. E1-E3 each trained an architecture end-to-end on ~14,000 windows drawn from 80 C-MAPSS training engines. This notebook trains **nothing**, calibrates **nothing**, and sees **no RUL label anywhere in its pipeline**: `amazon/chronos-t5-small` (Ansari et al., 2024) is a Transformer pretrained purely for general-purpose time-series forecasting, used here exactly as downloaded — frozen weights, zero gradient updates, no C-MAPSS-specific fine-tuning, no supervised head of any kind. The RUL estimate comes entirely from a hand-crafted, unsupervised heuristic applied to Chronos's own forecast uncertainty. The question: does a model that has never seen a turbofan sensor trace produce *forecast uncertainty* that tracks proximity to failure well enough to say something useful about industrial degradation — with zero labeled examples involved?

**Revision note.** An earlier version of this notebook fit a small linear regression head on a 20% calibration slice of the test engines and scored only the remaining 80 — an unfair comparison against E1-E3's full 100-engine evaluation, and not genuinely zero-shot besides (it still needed *some* labeled RUL values to fit the head). This version removes calibration and supervision entirely and scores on all 100 test engines, like every other experiment in this study.

**No training data used, and no labels used.** `train_FD001.txt` is never loaded. `src/data/cmapss_loader.get_cmapss` is called with `normalize=False` so no min-max scaler is fit on training-engine statistics either — Chronos performs its own internal instance/mean-scaling per input window, so it wants raw physical-unit sensor readings. The ground-truth `y_true` RUL values are used only at the very end, to *score* the heuristic — never to fit it.

**Reproducibility contract:** global seed fixed to 42 via `src/utils/seed.py::set_global_seed` (called before any data loading, model loading, or the one stochastic component — Chronos's sampling-based forecast — runs), CPU-only execution, deterministic sequential engine loop. Restart the kernel and *Run All* to reproduce every number in this notebook exactly.


## 0. Setup

`chronos-forecasting` is a project dependency, added to `requirements.txt` (pinned to `2.3.1`, matching every other package in that file) rather than installed inline here — this notebook assumes `pip install -r requirements.txt` has already been run, consistent with how E1-E3 assume `torch`/`sklearn`/etc. are already present.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm


def _find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing configs/paths.py is found."""
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "paths.py").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the AMMPM project root (no configs/paths.py found above "
        f"{start}). Launch Jupyter from the project root or notebooks/ directory."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.paths import RESULTS_DIR, ensure_dirs  # noqa: E402
from src.data.cmapss_loader import get_cmapss  # noqa: E402
from src.utils.seed import set_global_seed  # noqa: E402

SEED = 42
set_global_seed(SEED)
ensure_dirs()

from chronos import ChronosPipeline  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")
print(f"Global seed:  {SEED}")
print(f"Torch version: {torch.__version__}")


Project root: /home/bruce-wayne-2005/industrial-ai-project
Global seed:  42
Torch version: 2.13.0+cu130


## 1. Data: NASA C-MAPSS FD001 — test set only

We load through the shared `get_cmapss` loader for consistency with E1-E3 (RUL capping at 125 cycles, same 100 test engines, same ground-truth labels), but with `normalize=False` and the training split discarded immediately — the model never sees `train_FD001.txt`'s sensor trajectories, and no statistic derived from it (not even a min/max) enters this notebook.


In [2]:
MAX_RUL = 125
DATA = get_cmapss(fd_num=1, max_rul=MAX_RUL, normalize=False)
test_df = DATA["test_df"]
feature_columns = DATA["feature_columns"]
del DATA  # drop the reference so train_df/X_train/y_train are never touched below

print(f"FD001 test: {test_df['unit_number'].nunique()} engines, {len(test_df)} rows")
print(f"Feature channels ({len(feature_columns)}): {feature_columns}")


FD001 test: 100 engines, 13096 rows
Feature channels (24): ['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


## 2. Context windows: last 30 cycles per engine

Chronos needs no training windows — it only needs, per engine, the raw context it will extrapolate from. We reuse the same `SEQUENCE_LENGTH=30` and the same `mode="last"` windowing logic as E1-E3's test-time evaluation (one window per engine, ending at the engine's final recorded cycle), so the *definition* of "last 30 cycles" and the ground-truth RUL target are identical across all four experiments — the comparison in §6 is apples-to-apples, all four notebooks now scoring the same 100 engines.


In [3]:
SEQUENCE_LENGTH = 30  # cycles of context handed to Chronos


def build_last_window(df: pd.DataFrame, feature_cols, window: int):
    """One trailing window (and its ground-truth RUL) per engine."""
    X_list, y_list = [], []
    for _, group in df.groupby("unit_number"):
        group = group.sort_values("time_in_cycles")
        feats = group[feature_cols].to_numpy(dtype=np.float32)
        rul = group["RUL"].to_numpy(dtype=np.float32)
        n = len(group)

        if n < window:
            # Left-pad short trajectories by repeating the earliest reading
            # (none of FD001's 100 test engines are this short, but kept for
            # consistency/safety with the other C-MAPSS subsets).
            pad = np.repeat(feats[:1], window - n, axis=0)
            feats = np.concatenate([pad, feats], axis=0)

        X_list.append(feats[-window:])
        y_list.append(rul[-1])

    return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.float32)


X_context, y_true = build_last_window(test_df, feature_columns, SEQUENCE_LENGTH)
print(f"Context windows: {X_context.shape}  (engines, cycles, channels)")
print(f"Ground-truth RUL: {y_true.shape}")


Context windows: (100, 30, 24)  (engines, cycles, channels)
Ground-truth RUL: (100,)


## 3. Zero-shot forecasting with Chronos: probabilistic, not point

For each engine, all 24 channels' 30-cycle contexts are handed to `amazon/chronos-t5-small` as one batch (channel-independent forecasting, same principle as E3's PatchTST, but here there is no shared encoder being trained — the weights are exactly as pretrained). Chronos is a *probabilistic* forecaster: instead of one number, it samples `NUM_SAMPLES=50` possible next-step trajectories per channel. This time we keep the whole sample distribution rather than collapsing it to a mean, because the signal this notebook bets on is the **spread** of that distribution, not its center — see §4.


In [4]:
CHRONOS_MODEL_ID = "amazon/chronos-t5-small"
NUM_SAMPLES = 50
FORECAST_HORIZON = 1
INTERVAL_QUANTILES = (0.1, 0.9)  # central 80% prediction interval

pipeline = ChronosPipeline.from_pretrained(CHRONOS_MODEL_ID, device_map="cpu", dtype=torch.float32)

n_engines = X_context.shape[0]
n_channels = len(feature_columns)
raw_interval_width = np.zeros((n_engines, n_channels), dtype=np.float32)

quantile_tensor = torch.tensor(INTERVAL_QUANTILES, dtype=torch.float32)

for i in tqdm(range(n_engines), desc="Chronos zero-shot forecast per engine"):
    # (channels, context_length): one univariate series per sensor/op-setting channel
    context_batch = torch.from_numpy(X_context[i].T).to(torch.float32)
    forecast = pipeline.predict(
        context_batch, prediction_length=FORECAST_HORIZON, num_samples=NUM_SAMPLES
    ).squeeze(-1)  # (channels, num_samples)

    q_lo, q_hi = torch.quantile(forecast, quantile_tensor, dim=1)  # each: (channels,)
    raw_interval_width[i] = (q_hi - q_lo).numpy()

print(f"Raw interval widths: {raw_interval_width.shape}  (engines, channels)")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Chronos zero-shot forecast per engine:   0%|          | 0/100 [00:00<?, ?it/s]

Raw interval widths: (100, 24)  (engines, channels)


## 4. A fully unsupervised RUL proxy: forecast uncertainty, min-max scaled

**The hypothesis.** As a machine's condition degrades, its sensor dynamics tend to become less regular — noisier, more erratic, harder for any forecaster (including one that has never seen a turbofan before) to pin down. A wider Chronos prediction interval for a given engine is read as *the model itself signaling more uncertainty about what happens next* — and under this hypothesis, more forecast uncertainty means less remaining useful life. No RUL label is involved in forming this signal; it falls out of Chronos's own probabilistic forecast.

**Two min-max passes, both fit only on the forecast outputs themselves — never on labels:**

1. **Per-channel normalization.** Raw interval widths live on wildly different scales across channels (a multi-thousand-unit sensor's interval is naturally wider in absolute terms than a near-zero-mean operating-setting's, with no connection to degradation). Each channel's 100 raw widths are independently min-max scaled to `[0, 1]` *within that channel* before anything is combined across channels — otherwise the aggregate would simply track whichever sensor happens to have the largest physical units, exactly the scale-domination failure that broke the previous (regression-based) version of this notebook.
2. **Cross-channel aggregation, then final min-max to `[0, 125]`.** The 24 per-channel-normalized widths are averaged into one uncertainty score per engine, then that 100-engine population is itself min-max scaled — inverted, since *more* uncertainty should map to *less* RUL — directly onto `[0, MAX_RUL]`. This guarantees every prediction is bounded in the valid RUL range by construction (the most-uncertain engine lands at exactly 0, the least-uncertain at exactly 125), so no clipping or regularization hacks are needed anywhere in this version.


In [5]:
# Step 1: per-channel min-max normalization across the 100-engine population
channel_min = raw_interval_width.min(axis=0, keepdims=True)
channel_max = raw_interval_width.max(axis=0, keepdims=True)
channel_range = channel_max - channel_min
channel_range[channel_range == 0] = 1.0  # guard channels with zero width variation across engines

normalized_width = (raw_interval_width - channel_min) / channel_range  # (engines, channels), each column in [0, 1]

# Step 2: average across channels -> one uncertainty score per engine, then invert-scale to [0, MAX_RUL]
aggregate_uncertainty = normalized_width.mean(axis=1)  # (engines,)

u_min, u_max = aggregate_uncertainty.min(), aggregate_uncertainty.max()
u_range = u_max - u_min if u_max > u_min else 1.0  # guard the degenerate all-equal case

y_pred = MAX_RUL * (u_max - aggregate_uncertainty) / u_range

print(f"Aggregate uncertainty range: [{aggregate_uncertainty.min():.4f}, {aggregate_uncertainty.max():.4f}]")
print(f"Predicted RUL range:         [{y_pred.min():.2f}, {y_pred.max():.2f}] cycles (bounded in [0, {MAX_RUL}] by construction)")


Aggregate uncertainty range: [0.1368, 0.3883]
Predicted RUL range:         [0.00, 125.00] cycles (bounded in [0, 125] by construction)


## 5. Evaluation: RMSE, MAE, and the PHM08 asymmetric score

Same scoring function and same asymmetric-risk rationale as E1-E3: for error `d = predicted - actual`, early errors (`d < 0`) cost `exp(-d/13) - 1`, late errors (`d >= 0`) cost the steeper `exp(d/10) - 1`. Computed here on **all 100 test engines** — there is no calibration split to exclude anything from scoring.


In [6]:
def phm_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """PHM08 challenge asymmetric scoring function (lower is better)."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(np.sum(scores))


rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
mae = float(np.mean(np.abs(y_pred - y_true)))
phm = phm_score(y_true, y_pred)

print(f"Test RMSE:      {rmse:.3f} cycles")
print(f"Test MAE:       {mae:.3f} cycles")
print(f"Test PHM Score: {phm:.3f}  (n={len(y_true)} engines)")


Test RMSE:      45.374 cycles
Test MAE:       39.160 cycles
Test PHM Score: 23919.400  (n=100 engines)


## 6. Persisting results


In [7]:
metrics = {
    "experiment_id": "E4",
    "model": "chronos_zeroshot",
    "dataset": "cmapss",
    "fd_subset": "FD001",
    "seed": SEED,
    "chronos_model_id": CHRONOS_MODEL_ID,
    "hyperparameters": {
        "context_length": SEQUENCE_LENGTH,
        "forecast_horizon": FORECAST_HORIZON,
        "num_samples": NUM_SAMPLES,
        "interval_quantiles": list(INTERVAL_QUANTILES),
        "rul_mapping": (
            "unsupervised: per-channel min-max normalized Chronos forecast "
            "interval width, averaged across channels, inverse min-max "
            "scaled to [0, max_rul] using the forecast population itself "
            "(no calibration engines, no labels used to fit the mapping)"
        ),
        "max_rul": MAX_RUL,
    },
    "metrics": {
        "rmse": rmse,
        "mae": mae,
        "phm_score": phm,
    },
    "n_test_engines": int(len(y_true)),
}

results_path = RESULTS_DIR / "E4_chronos_zeroshot_fd001.json"
with open(results_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to {results_path}")
metrics


Saved metrics to /home/bruce-wayne-2005/industrial-ai-project/results/E4_chronos_zeroshot_fd001.json


{'experiment_id': 'E4',
 'model': 'chronos_zeroshot',
 'dataset': 'cmapss',
 'fd_subset': 'FD001',
 'seed': 42,
 'chronos_model_id': 'amazon/chronos-t5-small',
 'hyperparameters': {'context_length': 30,
  'forecast_horizon': 1,
  'num_samples': 50,
  'interval_quantiles': [0.1, 0.9],
  'rul_mapping': 'unsupervised: per-channel min-max normalized Chronos forecast interval width, averaged across channels, inverse min-max scaled to [0, max_rul] using the forecast population itself (no calibration engines, no labels used to fit the mapping)',
  'max_rul': 125},
 'metrics': {'rmse': 45.373600006103516,
  'mae': 39.159828186035156,
  'phm_score': 23919.400390625},
 'n_test_engines': 100}

## Persisting per-engine predictions

Saved alongside the aggregate metrics so downstream analysis/visualization notebooks (e.g. `EX_visualizations.ipynb`) can read per-engine true/predicted RUL directly, instead of re-running training.


In [8]:
predictions_path = RESULTS_DIR / "E4_chronos_zeroshot_fd001_predictions.npz"
np.savez(predictions_path, y_true=y_true, y_pred=y_pred)

print(f"Saved per-engine predictions to {predictions_path}")


Saved per-engine predictions to /home/bruce-wayne-2005/industrial-ai-project/results/E4_chronos_zeroshot_fd001_predictions.npz


## 7. Comparison to E1 (LSTM), E2 (Transformer), and E3 (PatchTST)

All four experiments now score on the same 100 FD001 test engines, so this table is a fair, matched-sample comparison — the only remaining difference is how each model gets to its RUL number: E1-E3 learn it end-to-end from ~14,000 labeled training windows, E4 derives it from a fixed unsupervised heuristic applied to a frozen, never-fine-tuned forecaster.


In [9]:
result_files = {
    "E1 (LSTM)": RESULTS_DIR / "E1_lstm_fd001.json",
    "E2 (Transformer)": RESULTS_DIR / "E2_transformer_fd001.json",
    "E3 (PatchTST)": RESULTS_DIR / "E3_patchtst_fd001.json",
    "E4 (Chronos zero-shot)": RESULTS_DIR / "E4_chronos_zeroshot_fd001.json",
}

rows = {}
for label, path in result_files.items():
    if path.exists():
        with open(path) as f:
            rows[label] = json.load(f)["metrics"]
    else:
        print(f"{label} results not found at {path} — run its notebook first for a full comparison.")

if rows:
    comparison = pd.DataFrame(rows).T
    print(comparison)

    trained_best_rmse = min(
        v["rmse"] for k, v in rows.items() if k != "E4 (Chronos zero-shot)"
    ) if any(k != "E4 (Chronos zero-shot)" for k in rows) else None

    if "E4 (Chronos zero-shot)" in rows and trained_best_rmse is not None:
        gap = rows["E4 (Chronos zero-shot)"]["rmse"] - trained_best_rmse
        print(f"\nChronos zero-shot RMSE is {gap:+.2f} cycles vs. the best trained baseline.")


                             rmse        mae     phm_score
E1 (LSTM)               40.532047  35.096893  18182.324219
E2 (Transformer)        14.618472  10.854953    412.316498
E3 (PatchTST)           14.833669  11.972316    385.594482
E4 (Chronos zero-shot)  45.373600  39.159828  23919.400391

Chronos zero-shot RMSE is +30.76 cycles vs. the best trained baseline.


## Summary

This notebook is the study's cleanest test of the time-series-foundation-model hypothesis: a pretrained forecaster, never shown a single C-MAPSS engine and never given a single RUL label, versus three architectures trained end-to-end on the real task with full supervision. Whatever gap remains between E4 and E1-E3 is the current cost of skipping *all* task-specific learning — the number future experiments (fine-tuned Chronos, richer unsupervised or lightly-supervised probes, cross-machine transfer) are trying to close. Results are persisted to `results/E4_chronos_zeroshot_fd001.json` for the ongoing cross-experiment comparison.
